# Time-series analysis with threshold classification — QuPath measurements

**Template** — edit the *Configuration* cell and run all cells.

Use when objects can be split into two morphology classes (e.g. Round vs elongated)
by a threshold on a single shape feature (typically Circularity). Tracks both mean
metric values and the fraction of objects in the shape class over time.

Input: QuPath TSV + `plate_map.csv`.
Output: per-well QC, threshold distribution plot, time-series of mean metrics and
shape fraction.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path().resolve().parents[1]  # repo root
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import hcs_analysis as hcs

## Configuration

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────────────────
EXP_DIR       = ROOT / "data" / "TODO_experiment"
RESULTS_TSV   = EXP_DIR / "Results" / "measurements.tsv"
PLATE_MAP_CSV = EXP_DIR / "plate_map.csv"
OUTPUT_DIR    = EXP_DIR / "Results"

CLASSIFICATION = None  # pre-filter by QuPath Classification label; None keeps all

# Metrics to track as per-object means over time
METRICS = ["Circularity", "Area px^2"]

# Threshold classification: objects with CIRCULARITY_COL < THRESHOLD → SHAPE_CLASS
CIRCULARITY_COL = "Circularity"
THRESHOLD       = 0.5
ROUND_CLASS     = "Round"
SHAPE_CLASS     = "Shape"

CONDITIONS_ORDER = ["Control", "TODO"]  # display order in plots
# ────────────────────────────────────────────────────────────────────────────

## 1. Load data

In [ ]:
df = hcs.load_qupath(RESULTS_TSV)
plate_map = hcs.load_plate_map(PLATE_MAP_CSV)
df = hcs.merge_plate_map(df, plate_map)

if CLASSIFICATION is not None:
    df = hcs.filter_by(df, Classification=CLASSIFICATION)

print(f"{len(df):,} objects  |  {df['Well'].nunique()} wells  |  "
      f"{df['elapsed_min'].nunique()} timepoints")
print("Conditions:", df["condition"].dropna().unique().tolist())
df.head(3)

## 2. QC — object counts per well

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
hcs.qc_counts(df, groupby="Well", ax=ax)
ax.set_title("Object count per well (all timepoints)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "QC_object_counts.pdf", format="pdf", bbox_inches="tight")

## 3. Explore threshold

Histogram of the classification feature across all objects and timepoints.
Adjust `THRESHOLD` in the Configuration cell until the split looks right.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[CIRCULARITY_COL].dropna(), bins=60, edgecolor="none", alpha=0.8)
ax.axvline(THRESHOLD, color="red", linestyle="--", label=f"threshold = {THRESHOLD}")
ax.set_xlabel(CIRCULARITY_COL)
ax.set_ylabel("object count")
ax.set_title(f"Distribution of {CIRCULARITY_COL} (all objects, all timepoints)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "circularity_distribution.pdf", format="pdf", bbox_inches="tight")

n_shape = (df[CIRCULARITY_COL] < THRESHOLD).sum()
print(f"{SHAPE_CLASS}: {n_shape:,} ({100 * n_shape / len(df):.1f}%)")
print(f"{ROUND_CLASS}: {len(df) - n_shape:,} ({100 * (len(df) - n_shape) / len(df):.1f}%)")

## 4. Classify and aggregate

The binary `is_shape` column (0/1) is averaged per well to give a shape fraction directly.

In [ ]:
df["is_shape"] = (df[CIRCULARITY_COL] < THRESHOLD).astype(float)

# Per-well means at each timepoint (is_shape mean = fraction)
df_well = hcs.to_well_means(df, metrics=METRICS + ["is_shape"])
df_well = df_well.rename(columns={"is_shape": "shape_fraction"})

# Per-biological-replicate means
track_cols = METRICS + ["shape_fraction"]
df_bio = (
    df_well
    .groupby(["sample", "condition", "elapsed_min"])[track_cols]
    .mean()
    .reset_index()
)
df_bio["elapsed_h"] = df_bio["elapsed_min"] / 60

print("Biological replicates per condition:")
print(df_bio.groupby("condition")["sample"].nunique())
df_bio.head(3)

## 5. Time-series — mean metrics

One line per condition; shaded band = 95 % bootstrapped CI across biological replicates.

In [ ]:
df_plot = df_bio[df_bio["condition"].isin(CONDITIONS_ORDER)].copy()

fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5))
if len(METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, METRICS):
    hcs.time_series(df_plot, y=metric, time_col="elapsed_h", ax=ax)
    ax.set_xlabel("Time (h)")
    ax.set_title(metric)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "metrics_timeseries.pdf", format="pdf", bbox_inches="tight")
plt.show()

## 6. Time-series — shape fraction

Fraction of objects classified as the shape class (circularity < threshold) per condition over time.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
hcs.time_series(df_plot, y="shape_fraction", time_col="elapsed_h", ax=ax)
ax.set_xlabel("Time (h)")
ax.set_ylabel(f"Fraction {SHAPE_CLASS}")
ax.set_ylim(0, 1)
ax.set_title(f"Fraction of {SHAPE_CLASS} objects over time (threshold = {THRESHOLD})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shape_fraction_timeseries.pdf", format="pdf", bbox_inches="tight")
plt.show()